# Project 1

We do a 7-class emotion classification on the ISEAR dataset. Our goal is to compare different representations and architectures with cross validation and Bayesian estimation.

## Imports

In [ ]:
# Imports
import sys
from pathlib import Path

root = Path.cwd().parent
sys.path.insert(0, str(root))

import part1

import datasets
import numpy as np
import re

from tokenizers import Tokenizer
from tabulate import tabulate
from collections import OrderedDict

from itertools import combinations
import pandas as pd

## Dataset

In [ ]:

dataset=datasets.load_dataset("gsri-18/ISEAR-dataset-complete") # we use ISEAR for the projects

In [ ]:
# Small dataset statistic cell

print("Names of the splits: ", dataset.keys(), "\n"*2)
print("Number of rows: ", dataset.num_rows, "\n"*2)
print("A single Dataset: ", dataset["train"]), "\n"*2
print("Names of the splits: ", dataset["train"].features, "\n"*2)
print("Column names: ", dataset["train"].column_names, "\n"*2)
print("Rows: ", dataset["train"].num_rows, "\n"*2)
print("One example as dict: ", dataset["train"][0], "\n"*2)
print("First 5 rows as a dict of lists: ", dataset["train"][:5], "\n"*2)
print("Whole column as a list: ", dataset["train"]["content"], "\n"*2)
print("Unique emotions: ", dataset["train"].unique("emotion"), "\n"*2)


In [ ]:
# normalize and preporcess text

def normalise_text(text: str) -> str:
    # any whitespace into single blank space
    return re.sub(r"\s+", " ", text).strip()

# when inspecting the dataset there were 2 empty columns, remove them (it is also visible on HF)
if "Unnamed: 2" in dataset["train"].column_names:
    dataset = dataset.remove_columns("Unnamed: 2")

print("Before preprocessing: ", dataset["train"][0]["content"], "\n")
dataset = dataset.map(lambda ex: {"content": normalise_text(ex["content"])})

print("After preprocessing: ", dataset["train"][0]["content"])
print("Docs still containing a newline:", sum("\n" in t for t in dataset["train"]["content"]))
print("Docs with double spaces:", sum("  " in t for t in dataset["train"]["content"]))


In [ ]:
agnews_outcome_space = part1.clslib.NamedClasses(['anger', 'disgust', 'fear', 'guilt', 'joy', 'sadness', 'shame'])
label_mapping = {name: i for i,name in enumerate(agnews_outcome_space)}
dataset = dataset.map(lambda example: {'emotion': label_mapping[example['emotion']]})


In [ ]:

corpus = dataset["train"]["content"]
len(corpus)
vocab_size = 1000
min_frequency = 2

results_folder = Path("./results")
results_folder.mkdir(parents=True, exist_ok=True)

tokenizer = part1.experiment.train_tokenizer(corpus, vocab_size, min_frequency)
tokenizer.save(f"{results_folder}/BPE_tokenizer_vs_{vocab_size}_mf_{min_frequency}.json")

#corpus_testing = dataset["test"]["content"]


In [ ]:
my_seeds = [0, 3, 5]
# my_seeds = [0] # to add less seeds which saves comp time

num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

for seed in my_seeds:
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(dataset["train"]["content"], dataset["train"]["emotion"])),
            k=num_folds,
            seed=seed
        )
    )

In [ ]:
my_classifiers = OrderedDict()

In [ ]:
# shared settings, so every system is trained with the same hyperparameters
loglin_args = {'max_iter': 100}
smlp_args = {
    'hidden_dim': 128,
    'dropout': 0.1,
    'batch_size': 64,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'num_epochs': 30,
}


In [ ]:
class Encoder:

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")

In [ ]:
class TokenIndicator(Encoder):

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(set(self.tokenizer.encode(x).ids))

        # A 0 vector with the same length as the vocabulary except in positions corresponding
        # to tokens appearing in the document
        vector = np.zeros(len(self))
        vector[unique_token_ids] = 1
        return vector

In [ ]:
token_indicator_encoder = TokenIndicator(tokenizer)
doc_repr = token_indicator_encoder(corpus[0])

In [ ]:
class BagOfWords(Encoder):
    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
    def __len__(self) -> int:
            """Dimensionality of the representation space"""
            return len(tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(self.tokenizer.encode(x).ids)

        # A 0 vector with the same length as the vocabulary except in positions corresponding
        # to tokens appearing in the document
        vector = np.zeros(len(self))
        for id in unique_token_ids:
            vector[id] += 1
        return vector

bow_encoder = BagOfWords(tokenizer)

# TF IDF

Implementation is based on https://medium.com/bitgrit-data-science-publication/tf-idf-from-scratch-in-python-ea587d003e9e

In [ ]:
def tf(word_counts: dict, document: list) -> dict:
    """Calculate term frequency of each word in a document."""
    tf_dict = {}
    corpus_count = max(len(document), 1)   # guard against empty documents

    for word, count in word_counts.items():
        tf_dict[word] = count / float(corpus_count)

    return tf_dict


def idf(documents: list, vocab) -> dict:
    """For each term, the (smoothed) inverse of the number of documents it appears in."""
    N = len(documents)
    idf_dict = dict.fromkeys(vocab, 0)

    for doc in documents:
        for word in set(doc):          # count each term once per document
            idf_dict[word] += 1

    for word, df in idf_dict.items():
        idf_dict[word] = np.log10((N + 1.0) / (df + 1.0))

    return idf_dict


def tfidf(doc_elements: dict, idfs: dict) -> dict:
    """TF * IDF per word given a single document."""
    tfidf_dict = {}

    for word, val in doc_elements.items():
        tfidf_dict[word] = val * idfs[word]

    return tfidf_dict


class TfIdf(Encoder):

    def __init__(self, tokenizer: Tokenizer, fit_corpus: list):
        self.tokenizer = tokenizer
        self.vocab = range(tokenizer.get_vocab_size())   # token ids are our "words"
        self.idfs = idf([tokenizer.encode(text).ids for text in fit_corpus], self.vocab)

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> 'np.array':
        document = self.tokenizer.encode(x).ids

        word_counts = dict.fromkeys(self.vocab, 0)
        for word in document:
            word_counts[word] += 1

        scores = tfidf(tf(word_counts, document), self.idfs)
        vector = np.array([scores[word] for word in self.vocab])

        norm = np.linalg.norm(vector)                    # L2-normalise (see note above)
        return vector / norm if norm > 0 else vector


tfidf_encoder = TfIdf(tokenizer, corpus)

vec = tfidf_encoder(corpus[0])
print(vec.shape, "non-zero entries:", np.count_nonzero(vec))
top = np.argsort(vec)[::-1][:5]
print([(tokenizer.id_to_token(int(i)), round(float(vec[i]), 3)) for i in top])


## Runs

In [ ]:
%%time
my_classifiers['logistic_indicator-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_indicator-enc/seed-{seed}"
    my_classifiers['logistic_indicator-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=token_indicator_encoder,
            model_option='logistic',
            save_to=run_folder,
            loglin_args=loglin_args,
        )
    )

In [ ]:
%%time
my_classifiers['logistic_bow-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_bow-enc/seed-{seed}"
    my_classifiers['logistic_bow-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=bow_encoder,
            model_option='logistic',
            save_to=run_folder,
            loglin_args=loglin_args,
        )
    )

In [ ]:
%%time
my_classifiers['logistic_tfidf-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_tfidf-enc/seed-{seed}"
    my_classifiers['logistic_tfidf-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=tfidf_encoder,
            model_option='logistic',
            save_to=run_folder,
            loglin_args=loglin_args,
        )
    )

In [ ]:
%%time
my_classifiers['mlp_indicator-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_indicator-enc/seed-{seed}"
    my_classifiers['mlp_indicator-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=token_indicator_encoder,
            model_option='mlp',
            save_to=run_folder,
            smlp_args=smlp_args,
        )
    )

In [ ]:
%%time
my_classifiers['mlp_bow-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_bow-enc/seed-{seed}"
    my_classifiers['mlp_bow-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=bow_encoder,
            model_option='mlp',
            save_to=run_folder,
            smlp_args=smlp_args,
        )
    )

In [ ]:
%%time
my_classifiers['mlp_tfidf-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_tfidf-enc/seed-{seed}"
    my_classifiers['mlp_tfidf-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=tfidf_encoder,
            model_option='mlp',
            save_to=run_folder,
            smlp_args=smlp_args,
        )
    )

In [ ]:
part1.analysis.plot_box_and_points(part1.analysis.make_data_frame(my_classifiers))

# Bayesian estimation and ROPE

In [ ]:
# First, a little helper, to gather all accuracies across runs and folds
def get_all_measurements_for(data, sys_name):
    return np.array([list(run_data['accuracies'].values()) for run_data in data[sys_name]])

# this will have shape [num_runs, num_folds]
get_all_measurements_for(my_classifiers, 'logistic_indicator-enc')

In [ ]:


ROPE = 0.005 

def compare_all_pairs(classifiers, rope_size=ROPE, runs=1, plot=False):
    rows = []
    for name_a, name_b in combinations(classifiers.keys(), 2):
        posterior = part1.analysis.BenavoliCorrelatedT(
            get_all_measurements_for(classifiers, name_a).flatten(),
            get_all_measurements_for(classifiers, name_b).flatten(),
            runs=runs,
            names=[name_a, name_b],
        )
        p_a, p_tie, p_b = posterior.probs(rope_size)
        rows.append({
            "A": name_a,
            "B": name_b,
            "mean diff (B-A)": posterior.mean_diff,
            "P(A wins)": p_a,
            "P(tie / in ROPE)": p_tie,
            "P(B wins)": p_b,
            "effect size": posterior.effect_size,
        })
        if plot:
            posterior.plot(rope_size)
    return pd.DataFrame(rows)

results_df = compare_all_pairs(my_classifiers, rope_size=ROPE, runs=len(my_seeds))
results_df.style.format({
    "mean diff (B-A)": "{:+.4f}",
    "P(A wins)": "{:.3f}",
    "P(tie / in ROPE)": "{:.3f}",
    "P(B wins)": "{:.3f}",
    "effect size": "{:.2f}",
})


In [ ]:
posterior = part1.analysis.BenavoliCorrelatedT(
    get_all_measurements_for(my_classifiers, 'logistic_bow-enc').flatten(),
    get_all_measurements_for(my_classifiers, 'mlp_bow-enc').flatten(),
    runs=len(my_seeds),
    names=['logistic_bow-enc', 'mlp_bow-enc'],
)
_ = posterior.plot(ROPE)
